# Developer Sentiment on Open-Source AI Models (2023–2026)

A quantitative corpus study analyzing GitHub issue discourse across five major open-source AI/ML repositories to trace sentiment trends toward seven open-weight model families: Llama, Qwen, Gemma, DeepSeek, Phi, Kimi, and GLM, between 2023 and 2026.

**Data source:** GitHub REST Search API  
**Repositories:** `pytorch/pytorch`, `huggingface/transformers`, `tensorflow/tensorflow`, `ollama/ollama`, `AUTOMATIC1111/stable-diffusion-webui`  
**Method:** Lexicon-based sentiment scoring (VADER) + keyword-based model tagging

---

In [ ]:
import requests
import pandas as pd
import time
from getpass import getpass

# Enter my GitHub token securely  
token = getpass("Enter your GitHub token: ")
headers = {"Authorization": f"token {token}"}

# Quick sanity check that auth works
resp = requests.get("https://api.github.com/user", headers=headers, timeout=10)
print(resp.status_code, resp.json().get("login"))

In [3]:
repos = [
    "pytorch/pytorch",
    "huggingface/transformers",
    "tensorflow/tensorflow",
    "ollama/ollama",
    "AUTOMATIC1111/stable-diffusion-webui"
]

model_config = {
    "Llama":    {"keywords": ["llama 2", "llama2", "llama-2", "llama 3", "llama3", "llama-3",
                               "llama 4", "llama4", "llama-4"]},
    "Phi":      {"keywords": ["phi-2", "phi2", "phi-3", "phi3", "phi-4", "phi4"]},
    "Qwen":     {"keywords": ["qwen", "qwen2", "qwen2.5", "qwen3", "qwen-vl", "qwen-coder"]},
    "Gemma":    {"keywords": ["gemma 2", "gemma2", "gemma-2", "gemma 3", "gemma3", "gemma-3",
                               "gemma 4", "gemma4", "gemma-4"]},
    "DeepSeek": {"keywords": ["deepseek", "deepseek-v2", "deepseek-v3", "deepseek-r1",
                               "deepseek v3.2", "deepseek-v4"]},
    "Kimi":     {"keywords": ["kimi k2", "kimi-k2", "kimi k2.5", "kimi k2.6", "kimi k2.7"]},
    "GLM":      {"keywords": ["glm-5", "glm5", "glm-5.1", "glm-5.2"]},
    "Open Source (general)": {"keywords": ["open source", "open-source", "open weight", "open-weight"]}
}

years = [
    ("2023-01-01", "2023-12-31"),
    ("2024-01-01", "2024-12-31"),
    ("2025-01-01", "2025-12-31"),
    ("2026-01-01", "2026-07-01")
]

In [5]:
all_issues = []
error_log = []

for repo in repos:
    for start, end in years:
        for page in range(1, 11):  # up to 1000/repo/year 
            url = "https://api.github.com/search/issues"
            params = {
                "q": f"repo:{repo} type:issue created:{start}..{end}",
                "per_page": 100,
                "page": page,
                "sort": "created",
                "order": "asc"
            }
            resp = requests.get(url, headers=headers, params=params, timeout=10)

            if resp.status_code != 200:
                print(f"ERROR {resp.status_code} for {repo} {start}-{end} page {page}")
                error_log.append((repo, start, end, page, resp.status_code, resp.text))
                time.sleep(5)
                continue

            items = resp.json().get("items", [])
            if not items:
                break

            all_issues.extend(items)
            time.sleep(2)

    print(f"Done: {repo} — running total: {len(all_issues)}")

print("\nFINAL TOTAL:", len(all_issues))

Done: pytorch/pytorch — running total: 4000
Done: huggingface/transformers — running total: 7751
Done: tensorflow/tensorflow — running total: 11236
Done: ollama/ollama — running total: 15236
Done: AUTOMATIC1111/stable-diffusion-webui — running total: 17307

FINAL TOTAL: 17307


In [6]:
df = pd.DataFrame(all_issues)
print(df.shape)

# Save raw data right away so I never have to re-run the API calls
df.to_csv("github_issues_raw.csv", index=False)
print("Saved raw data.")

(17307, 35)
Saved raw data.


In [7]:
df["repo_source"] = df["html_url"].apply(lambda x: "/".join(x.split("/")[3:5]))
df["created_at"] = pd.to_datetime(df["created_at"])
df["year"] = df["created_at"].dt.year
df["full_text"] = (df["title"].fillna("") + " " + df["body"].fillna("")).str.lower()

print(df["year"].value_counts().sort_index())
print(df["repo_source"].value_counts())

year
2023    5000
2024    4856
2025    4103
2026    3348
Name: count, dtype: int64
repo_source
pytorch/pytorch                         4000
ollama/ollama                           4000
huggingface/transformers                3751
tensorflow/tensorflow                   3485
AUTOMATIC1111/stable-diffusion-webui    2071
Name: count, dtype: int64


In [8]:
def find_matches(text):
    matches = []
    for model, cfg in model_config.items():
        if any(kw.lower() in text for kw in cfg["keywords"]):
            matches.append(model)
    return matches

df["matched_models"] = df["full_text"].apply(find_matches)
df["num_matches"] = df["matched_models"].apply(len)

print(df["num_matches"].value_counts())
print(df[df["num_matches"] > 0]["matched_models"].explode().value_counts())

num_matches
0    15043
1     2017
2      203
3       31
4        8
5        5
Name: count, dtype: int64
matched_models
Qwen                     785
Llama                    738
DeepSeek                 399
Gemma                    288
Open Source (general)    231
Phi                       99
GLM                       17
Kimi                      16
Name: count, dtype: int64


In [9]:
qwen_examples = df[df["matched_models"].apply(lambda x: "Qwen" in x)][["title", "repo_source", "year"]].sample(5)
print(qwen_examples)

                                                   title  \
6082   use_liger_kernel requires much more GPU memory...   
7535   `Qwen3VLVisionPatchEmbed.proj` (`nn.Conv3d` wi...   
14590  Model request for Qwen3.5-397B-A17B local version   
7510   Proposal: add sdpa_memeff attn_implementation ...   
14341                                Error loading model   

                    repo_source  year  
6082   huggingface/transformers  2025  
7535   huggingface/transformers  2026  
14590             ollama/ollama  2026  
7510   huggingface/transformers  2026  
14341             ollama/ollama  2026  


In [10]:
print(df.loc[14341, "title"])
print(df.loc[14341, "body"][:500])

Error loading model
### What is the issue?

Finetuned qwen3-VL via LlamaFactory, converted from safetensors to gguf via llama.cpp, creating model in ollama is ok, but when i try to run model i got error, pls help

Others models include Qwen3-vl works fine

### Relevant log output

```shell
Jan 22 05:53:11 ai-nvidia-app01 ollama[4135940]: ggml_backend_cuda_device_get_memory device GPU-f96ee413-bbeb-db0e-f5b5-31cdf6e15749 utilizing NVML memory reporting free: 16721838080 total: 25757220864
Jan 22 05:53:11 ai-nvidia-a


In [11]:
llama_examples = df[df["matched_models"].apply(lambda x: "Llama" in x)][["title", "body", "repo_source", "year"]].sample(3)
for idx, row in llama_examples.iterrows():
    print(f"--- {row['repo_source']} ({row['year']}) ---")
    print(row["title"])
    print(str(row["body"])[:500])
    print()

--- ollama/ollama (2023) ---
Error: llama runner exited
Using mistral and llama2 with ollama, I received the following error message: `Error: llama runner exited, you may not have enough available memory to run this model?`.

The `README.md` states that at least 16GB of RAM is required to run 7B models, which is met by my workstation specifications.

--- huggingface/transformers (2025) ---
How can we use CPU offloading when using AutoModelForCausalLM and THUDM/cogvlm2-llama3-chat-19B
It is working great with below however still not sufficient. Uses around 16 GB VRAM

I want to further lower the requirement if possible

How can I achieve that?

model path is : `THUDM/cogvlm2-llama3-chat-19B`

```
                    model = AutoModelForCausalLM.from_pretrained(
                        MODEL_PATH,
                        torch_dtype=TORCH_TYPE,
                        trust_remote_code=True,
                        quantization_config=BitsAndBytesConfig(load_in_4bit=True),
   

--- huggi

In [12]:
import os
print(os.listdir())

['.ipynb_checkpoints', 'AAAL proposal.docx', 'Claude .docx', 'Github_7_models.ipynb', 'github_ai_issues_2023_2026.csv', 'github_issues_raw.csv', 'Github_new_pulls-v1.ipynb', 'Github_new_pulls.ipynb', 'Hugging face dataset.ipynb', 'Steps to collect data.docx', '~$AL proposal.docx', '~$laude .docx']


In [13]:
df.to_csv("github_issues_tagged.csv", index=False)
print("Saved:", df.shape)

Saved: (17307, 40)


In [21]:
gitignore_content = """.ipynb_checkpoints/
__pycache__/
*.pyc
.env
*.docx
~$*
"""

with open(".gitignore", "w") as f:
    f.write(gitignore_content)

print("Created .gitignore")

Created .gitignore


In [23]:
import os
print(os.listdir())

with open(".gitignore") as f:
    print(f.read())

['.gitignore', '.ipynb_checkpoints', 'AAAL proposal.docx', 'Claude .docx', 'github_issues_raw.csv', 'github_issues_tagged.csv', 'Github_openmodel_sentiment.ipynb', 'Github_sentiment_datasets.ipynb', 'Individual paper proposal.docx', 'sentiment_by_model_year_filtered.png', 'sentiment_by_year.png', '~$dividual paper proposal.docx', '~$laude .docx']
.ipynb_checkpoints/
__pycache__/
*.pyc
.env
*.docx
~$*



In [24]:
readme_content = """# Developer Sentiment on Open-Source AI Models (2023–2026)

A quantitative corpus study analyzing GitHub issue discourse across five major 
open-source AI/ML repositories to trace sentiment trends toward six open-weight 
model families (Llama, Qwen, Gemma, DeepSeek, Kimi, GLM) between 2023 and 2026.

## Data Collection
Issues were collected via the GitHub REST Search API from:
- pytorch/pytorch
- huggingface/transformers
- tensorflow/tensorflow
- ollama/ollama
- AUTOMATIC1111/stable-diffusion-webui

## Method
- Sentiment scoring via VADER
- Model-family tagging via keyword matching
- Aggregation by year and model family

## Files
- `Github_sentiment_datasets.ipynb` — data collection, tagging, and sentiment analysis
- `github_issues_tagged.csv` — final tagged dataset

## Requirements
See `requirements.txt`
"""

with open("README.md", "w") as f:
    f.write(readme_content)

print("Created README.md")

Created README.md


In [25]:
requirements_content = """pandas
requests
vaderSentiment
matplotlib
"""

with open("requirements.txt", "w") as f:
    f.write(requirements_content)

print("Created requirements.txt")

Created requirements.txt
